# Audit Visualization Notebook

This notebook visualizes outputs from:
- `outputs/coverage_audits`
- `outputs/annotation_audits`
- `outputs/disparity_audits`

Plots and tables are organized by research question:
- **RQ1** — Do benchmarks have lower dogwhistle coverage as coding sophistication increases (L1→L4)?
- **RQ2** — When dogwhistle terms appear, are they correctly labeled hateful, and does this vary by group and level?
- **RQ3** — Do benchmarks satisfy equalized odds across target groups?

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use('ggplot')

In [ ]:
WORKDIR = Path.cwd()
if not (WORKDIR / 'outputs').exists():
    WORKDIR = WORKDIR.parent

coverage_dir = WORKDIR / 'outputs' / 'coverage_audits'
annotation_dir = WORKDIR / 'outputs' / 'annotation_audits'
disparity_dir = WORKDIR / 'outputs' / 'disparity_audits'
viz_output_dir = WORKDIR / 'outputs' / 'audit_visualizations'
viz_output_dir.mkdir(parents=True, exist_ok=True)

def read_tsv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f'Missing required file: {path}')
    return pd.read_csv(path, sep='\t')

audit_metrics = read_tsv(coverage_dir / 'audit_metrics.tsv')
audit_detailed = read_tsv(coverage_dir / 'audit_detailed.tsv')
audit_matches = read_tsv(coverage_dir / 'audit_matches.tsv')
annotation_quality = read_tsv(annotation_dir / 'annotation_quality.tsv')
case_breakdown = read_tsv(annotation_dir / 'case_breakdown.tsv')
form_labeling = read_tsv(annotation_dir / 'form_labeling_detail.tsv')
coverage_disparity = read_tsv(disparity_dir / 'coverage_disparity.tsv')
annotation_disparity = read_tsv(disparity_dir / 'annotation_disparity.tsv')
cross_level_consistency = read_tsv(disparity_dir / 'cross_level_consistency.tsv')

# Define self-referential categories from form-level type annotations.
self_ref_mask = form_labeling['type'].fillna('').astype(str).str.contains('self-referential', case=False)
self_ref_rows = form_labeling[self_ref_mask].copy()
self_ref_pairs = set(
    zip(
        self_ref_rows['taxonomy_level'].astype(str),
        self_ref_rows['target'].astype(str),
    )
)
self_ref_triples = set(
    zip(
        self_ref_rows['taxonomy_level'].astype(str),
        self_ref_rows['target'].astype(str),
        self_ref_rows['dogwhistle'].astype(str),
    )
)

def exclude_self_ref_pairs(df: pd.DataFrame, level_col: str, target_col: str) -> pd.DataFrame:
    key = list(zip(df[level_col].astype(str), df[target_col].astype(str)))
    mask = [k not in self_ref_pairs for k in key]
    return df.loc[mask].copy()

def exclude_self_ref_triples(df: pd.DataFrame, level_col: str, target_col: str, dogwhistle_col: str) -> pd.DataFrame:
    key = list(zip(df[level_col].astype(str), df[target_col].astype(str), df[dogwhistle_col].astype(str)))
    mask = [k not in self_ref_triples for k in key]
    return df.loc[mask].copy()

# Build filtered views used by all visualizations.
audit_metrics_viz = exclude_self_ref_pairs(audit_metrics, 'taxonomy_level', 'target')
audit_detailed_viz = exclude_self_ref_triples(audit_detailed, 'taxonomy_level', 'target', 'dogwhistle')
audit_matches_viz = exclude_self_ref_triples(audit_matches, 'taxonomy_level', 'target', 'dogwhistle')
annotation_quality_viz = exclude_self_ref_pairs(annotation_quality, 'taxonomy_level', 'target')
case_breakdown_viz = exclude_self_ref_pairs(case_breakdown, 'taxonomy_level', 'target')
form_labeling_viz = exclude_self_ref_triples(form_labeling, 'taxonomy_level', 'target', 'dogwhistle')

coverage_disparity_viz = coverage_disparity[
    ~coverage_disparity.apply(
        lambda row: (
            (str(row['taxonomy_level']), str(row['target_a'])) in self_ref_pairs
            or (str(row['taxonomy_level']), str(row['target_b'])) in self_ref_pairs
        ),
        axis=1,
    )
].copy()

annotation_disparity_viz = annotation_disparity[
    ~annotation_disparity.apply(
        lambda row: (
            (str(row['taxonomy_level']), str(row['target_a'])) in self_ref_pairs
            or (str(row['taxonomy_level']), str(row['target_b'])) in self_ref_pairs
        ),
        axis=1,
    )
].copy()

cross_level_consistency_viz = cross_level_consistency[
    ~cross_level_consistency.apply(
        lambda row: (
            (str(row['level_from']), str(row['target'])) in self_ref_pairs
            or (str(row['level_to']), str(row['target'])) in self_ref_pairs
        ),
        axis=1,
    )
].copy()

coarse_levels = sorted(audit_metrics_viz['taxonomy_level'].dropna().astype(str).unique().tolist())

def make_facet_axes(levels: list[str], ncols: int = 3, panel_w: float = 5.0, panel_h: float = 4.0):
    n = len(levels)
    ncols = max(1, min(ncols, n))
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel_w * ncols, panel_h * nrows))
    axes = np.atleast_1d(axes).reshape(nrows, ncols).flatten()
    for idx in range(len(axes)):
        if idx >= n:
            axes[idx].axis('off')
    return fig, axes

print('Loaded datasets:')
for name, df in [
    ('audit_metrics', audit_metrics),
    ('audit_detailed', audit_detailed),
    ('audit_matches', audit_matches),
    ('annotation_quality', annotation_quality),
    ('case_breakdown', case_breakdown),
    ('form_labeling', form_labeling),
    ('coverage_disparity', coverage_disparity),
    ('annotation_disparity', annotation_disparity),
    ('cross_level_consistency', cross_level_consistency),
]:
    print(f'  - {name}: {len(df)} rows')

print('\nRemoved self-referential rows for visualizations:')
for name, full_df, filtered_df in [
    ('audit_metrics', audit_metrics, audit_metrics_viz),
    ('audit_detailed', audit_detailed, audit_detailed_viz),
    ('audit_matches', audit_matches, audit_matches_viz),
    ('annotation_quality', annotation_quality, annotation_quality_viz),
    ('case_breakdown', case_breakdown, case_breakdown_viz),
    ('form_labeling', form_labeling, form_labeling_viz),
    ('coverage_disparity', coverage_disparity, coverage_disparity_viz),
    ('annotation_disparity', annotation_disparity, annotation_disparity_viz),
    ('cross_level_consistency', cross_level_consistency, cross_level_consistency_viz),
]:
    print(f'  - {name}: removed {len(full_df) - len(filtered_df)} rows')

print('Facet levels:', coarse_levels)

In [ ]:
# ---------------------------------------------------------------------------
# Category mapping: collapse fine-grained targets (e.g. 'race_black') into
# coarse groups ('race', 'gender', 'religion', 'sexuality', 'political',
# 'other').  The prefix before the first underscore is used as the coarse
# group label; values that match no known coarse prefix fall into 'other'.
# ---------------------------------------------------------------------------

COARSE_PREFIXES = {
    'race': 'race',
    'gender': 'gender',
    'religion': 'religion',
    'sexuality': 'sexuality',
    'political': 'political',
    'origin': 'origin',
    'disability': 'disability',
}

# Additional explicit overrides for persona labels used by the glossary
# pipeline (e.g. 'racist' → 'race', 'antisemitic' → 'religion').
TARGET_COARSE_OVERRIDE = {
    'racist': 'race',
    'anti-black': 'race',
    'anti-asian': 'race',
    'anti-latino': 'race',
    'white supremacist': 'race',
    'xenophobic': 'race',
    'antisemitic': 'religion',
    'islamophobic': 'religion',
    'transphobic': 'gender',
    'misogynistic': 'gender',
    'homophobic': 'sexuality',
    'anti-lgbtq': 'sexuality',
    'conservative': 'political',
    'anti-liberal': 'political',
    'liberal': 'political',
    'anti-vax': 'political',
    'climate change denier': 'political',
    'anti-gmo': 'political',
    'religious': 'religion',
}

MIN_MATCHES_FOR_RELIABLE_RATE = 5  # cells with n < this are flagged as unreliable


def coarse_category(target: str) -> str:
    """Return the coarse group for a target string."""
    t = str(target).strip().lower()
    # explicit override
    if t in TARGET_COARSE_OVERRIDE:
        return TARGET_COARSE_OVERRIDE[t]
    # prefix split (e.g. 'race_black' → 'race')
    prefix = t.split('_')[0]
    if prefix in COARSE_PREFIXES:
        return COARSE_PREFIXES[prefix]
    return 'other'


# Attach coarse_category to all metric frames used by later cells.
for _df in [
    audit_metrics_viz,
    annotation_quality_viz,
    case_breakdown_viz,
    form_labeling_viz,
]:
    if 'coarse_category' not in _df.columns:
        _df['coarse_category'] = _df['target'].apply(coarse_category)

for _df, col in [
    (coverage_disparity_viz, 'target_a'),
    (annotation_disparity_viz, 'target_a'),
]:
    if 'coarse_a' not in _df.columns:
        _df['coarse_a'] = _df['target_a'].apply(coarse_category)
    if 'coarse_b' not in _df.columns:
        _df['coarse_b'] = _df['target_b'].apply(coarse_category)

if 'coarse_category' not in cross_level_consistency_viz.columns:
    cross_level_consistency_viz['coarse_category'] = cross_level_consistency_viz['target'].apply(coarse_category)

print('Coarse categories found in audit_metrics_viz:',
      sorted(audit_metrics_viz['coarse_category'].unique()))

---
## RQ1 — Dogwhistle Coverage Across Taxonomy Levels (L1 → L4)

> *Do benchmarks have lower dogwhistle coverage as coding sophistication increases?*

Plots show **presence rate** (fraction of glossary dogwhistles found at least once) and **type coverage**
per coarse target-group category, faceted by taxonomy level.  A downward trend from L1 to L4 supports RQ1.

In [ ]:
# RQ1 Plot 1: Presence rate by taxonomy level, aggregated per coarse category
# Each panel = one coarse category; x-axis = taxonomy level; y-axis = mean presence rate

coarse_cats = sorted(audit_metrics_viz['coarse_category'].unique())

fig, axes = make_facet_axes(coarse_cats, ncols=3, panel_w=5.2, panel_h=4.2)

for idx, cat in enumerate(coarse_cats):
    ax = axes[idx]
    cat_df = audit_metrics_viz[audit_metrics_viz['coarse_category'] == cat]

    if cat_df.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{cat}: presence rate')
        ax.set_axis_off()
        continue

    agg = (
        cat_df.groupby('taxonomy_level', sort=True)
        .agg(
            mean_presence=('presence_rate', 'mean'),
            n_targets=('presence_rate', 'count'),
        )
        .reset_index()
    )
    # Flag levels with few targets
    agg['reliable'] = agg['n_targets'] >= MIN_MATCHES_FOR_RELIABLE_RATE

    colors = ['#2f6db5' if r else '#aaaaaa' for r in agg['reliable']]
    ax.bar(agg['taxonomy_level'].astype(str), agg['mean_presence'], color=colors)
    ax.set_ylim(0, 1.05)
    ax.set_title(f'{cat}: presence rate')
    ax.set_xlabel('Taxonomy level')
    ax.set_ylabel('Mean presence rate')

    # Annotate bars with sample size
    for _, row in agg.iterrows():
        ax.text(
            str(row['taxonomy_level']),
            row['mean_presence'] + 0.02,
            f"n={int(row['n_targets'])}",
            ha='center',
            fontsize=7,
            color='#555555' if row['reliable'] else 'red',
        )

fig.suptitle(
    'RQ1 — Mean presence rate per coarse category across taxonomy levels\n'
    '(grey = fewer than {} targets; self-referential excluded)'.format(MIN_MATCHES_FOR_RELIABLE_RATE),
    y=0.99,
    fontsize=14,
    fontweight='semibold',
)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(viz_output_dir / 'rq1_presence_rate_by_level.png', dpi=180)
plt.show()

In [ ]:
# RQ1 Plot 2: Top token-frequency targets per coarse category (faceted by level)

fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=5.6, panel_h=4.6)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    plot_df = (
        audit_metrics_viz[audit_metrics_viz['taxonomy_level'] == level]
        .sort_values('token_frequency', ascending=False)
        .head(10)
        .copy()
    )

    if plot_df.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{level}: token frequency')
        ax.set_axis_off()
        continue

    plot_df = plot_df.sort_values('token_frequency', ascending=True)
    bar_colors = ['#2f6db5' if c == 'race' else
                  '#e06c00' if c == 'religion' else
                  '#1b9e77' if c == 'gender' else
                  '#9467bd' if c == 'sexuality' else
                  '#d62728' if c == 'political' else '#888888'
                  for c in plot_df['coarse_category']]

    ax.barh(plot_df['target'], plot_df['token_frequency'], color=bar_colors)
    ax.set_title(f'{level}: token frequency')
    ax.set_xlabel('Token frequency')
    ax.set_ylabel('Target')

fig.suptitle(
    'RQ1 — Top token-frequency targets per level (self-referential excluded)',
    y=0.99,
    fontsize=14,
    fontweight='semibold',
)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(viz_output_dir / 'rq1_token_frequency_by_level.png', dpi=180)
plt.show()

In [ ]:
# RQ1 Plot 3: Presence rate vs type coverage (scatter; bubble size = token frequency)

norm = mpl.colors.Normalize(
    vmin=max(float(audit_metrics_viz['token_frequency'].min()), 1.0),
    vmax=max(float(audit_metrics_viz['token_frequency'].max()), 1.0),
)

fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=5.6, panel_h=4.6)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    level_df = audit_metrics_viz[audit_metrics_viz['taxonomy_level'] == level].copy()

    if level_df.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{level}: presence vs type coverage')
        ax.set_axis_off()
        continue

    sizes = np.clip(level_df['token_frequency'].to_numpy(), 1, None)
    sizes = 30 + 150 * (sizes / sizes.max())

    ax.scatter(
        level_df['presence_rate'],
        level_df['type_coverage'],
        s=sizes,
        c=level_df['token_frequency'],
        cmap='viridis',
        norm=norm,
        alpha=0.8,
        edgecolor='black',
        linewidth=0.3,
    )
    ax.set_title(f'{level}: presence vs type coverage')
    ax.set_xlabel('Presence rate')
    ax.set_ylabel('Type coverage')
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)

fig.suptitle(
    'RQ1 — Coverage quality by taxonomy level (self-referential excluded)',
    y=0.99,
    fontsize=14,
    fontweight='semibold',
)
fig.tight_layout(rect=[0, 0.10, 1, 0.95])

sm = mpl.cm.ScalarMappable(norm=norm, cmap='viridis')
sm.set_array([])
left = axes[0].get_position().x0
right = axes[min(1, len(coarse_levels)-1)].get_position().x1
cax = fig.add_axes([left, 0.055, right - left, 0.028])
cbar = fig.colorbar(sm, cax=cax, orientation='horizontal')
cbar.set_label('Token frequency')

fig.savefig(viz_output_dir / 'rq1_presence_vs_type_coverage.png', dpi=180)
plt.show()

In [ ]:
# RQ1 Table 1: Presence rate by (coarse_category × taxonomy_level)
# Cells with fewer than MIN_MATCHES_FOR_RELIABLE_RATE targets are flagged.

rq1_agg = (
    audit_metrics_viz
    .groupby(['coarse_category', 'taxonomy_level'], sort=True)
    .agg(
        mean_presence_rate=('presence_rate', 'mean'),
        mean_type_coverage=('type_coverage', 'mean'),
        total_token_frequency=('token_frequency', 'sum'),
        n_targets=('presence_rate', 'count'),
    )
    .reset_index()
)

rq1_agg['mean_presence_rate'] = rq1_agg['mean_presence_rate'].round(3)
rq1_agg['mean_type_coverage'] = rq1_agg['mean_type_coverage'].round(3)
rq1_agg['data_flag'] = rq1_agg['n_targets'].apply(
    lambda n: '⚠ insufficient data' if n < MIN_MATCHES_FOR_RELIABLE_RATE else ''
)

# Pivot for a compact view: coarse_category rows × taxonomy_level columns
rq1_pivot = rq1_agg.pivot_table(
    index='coarse_category',
    columns='taxonomy_level',
    values='mean_presence_rate',
    aggfunc='first',
)

# Append flag column to pivot
flag_pivot = rq1_agg.pivot_table(
    index='coarse_category',
    columns='taxonomy_level',
    values='n_targets',
    aggfunc='first',
).map(lambda n: '⚠' if pd.notna(n) and n < MIN_MATCHES_FOR_RELIABLE_RATE else '')

print('RQ1 Table 1 — Mean presence rate by coarse category × taxonomy level')
print(f'(⚠ = n < {MIN_MATCHES_FOR_RELIABLE_RATE} targets in cell; self-referential excluded)\n')
print(rq1_pivot.to_string())
print('\nData-sufficiency flags:')
print(flag_pivot.to_string())

# Export
rq1_agg.to_csv(viz_output_dir / 'rq1_table1_coverage_by_category_level.tsv', sep='\t', index=False)
print(f'\nExported → {viz_output_dir / "rq1_table1_coverage_by_category_level.tsv"}')

---
## RQ2 — Labeling Accuracy Across Groups and Taxonomy Levels

> *When dogwhistle terms appear, are they correctly labeled hateful,
> and does this vary by group and level?*

Plots show **correct-labeling rate** (Case A / (Case A + Case B)) and the complementary
**annotator-failure ratio** per coarse target-group category.

In [ ]:
# RQ2 Plot 1: Annotation quality rates per coarse category, faceted by taxonomy level

fig, axes = make_facet_axes(coarse_cats, ncols=3, panel_w=5.4, panel_h=4.4)

for idx, cat in enumerate(coarse_cats):
    ax = axes[idx]
    cat_df = annotation_quality_viz[annotation_quality_viz['coarse_category'] == cat]

    if cat_df.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{cat}: labeling rates')
        ax.set_axis_off()
        continue

    agg = (
        cat_df.groupby('taxonomy_level', sort=True)
        .agg(
            mean_correct=('correct_labeling_rate', 'mean'),
            mean_failure=('annotator_failure_ratio', 'mean'),
            total_matches=('total_matches', 'sum'),
        )
        .reset_index()
    )
    agg['reliable'] = agg['total_matches'] >= MIN_MATCHES_FOR_RELIABLE_RATE

    x = np.arange(len(agg))
    width = 0.38
    correct_colors = ['#1b9e77' if r else '#aaaaaa' for r in agg['reliable']]
    failure_colors = ['#d95f02' if r else '#cccccc' for r in agg['reliable']]
    ax.bar(x - width / 2, agg['mean_correct'], width=width,
           label='correct', color=correct_colors)
    ax.bar(x + width / 2, agg['mean_failure'], width=width,
           label='failure', color=failure_colors)
    ax.set_xticks(x)
    ax.set_xticklabels(agg['taxonomy_level'].astype(str))
    ax.set_ylim(0, 1.05)
    ax.set_title(f'{cat}: labeling rates')
    ax.set_xlabel('Taxonomy level')
    ax.set_ylabel('Rate')
    if idx == 0:
        ax.legend(frameon=False, fontsize=8)

fig.suptitle(
    'RQ2 — Labeling rates per coarse category across taxonomy levels\n'
    '(grey = insufficient matches; self-referential excluded)',
    y=0.99,
    fontsize=14,
    fontweight='semibold',
)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(viz_output_dir / 'rq2_labeling_rates_by_category.png', dpi=180)
plt.show()

In [ ]:
# RQ2 Plot 2: Annotation quality rates for top targets, faceted by taxonomy level

fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=6.2, panel_h=4.8)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    annot_plot = (
        annotation_quality_viz[annotation_quality_viz['taxonomy_level'] == level]
        .sort_values('total_matches', ascending=False)
        .head(8)
        .copy()
    )

    if annot_plot.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{level}: annotation quality')
        ax.set_axis_off()
        continue

    annot_plot['reliable'] = annot_plot['total_matches'] >= MIN_MATCHES_FOR_RELIABLE_RATE

    x = np.arange(len(annot_plot))
    width = 0.4
    ax.bar(
        x - width / 2,
        annot_plot['correct_labeling_rate'],
        width=width,
        label='correct',
        color='#1b9e77',
    )
    ax.bar(
        x + width / 2,
        annot_plot['annotator_failure_ratio'],
        width=width,
        label='failure',
        color='#d95f02',
    )
    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"{t}{'⚠' if not r else ''}" for t, r
         in zip(annot_plot['target'], annot_plot['reliable'])],
        rotation=50,
        ha='right',
    )
    ax.set_ylim(0, 1)
    ax.set_title(f'{level}: annotation quality')
    ax.set_ylabel('Rate')
    ax.legend(frameon=False, fontsize=8)

fig.suptitle(
    'RQ2 — Annotation quality by level (⚠ = n < {}; self-referential excluded)'.format(
        MIN_MATCHES_FOR_RELIABLE_RATE
    ),
    y=0.99,
    fontsize=14,
    fontweight='semibold',
)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(viz_output_dir / 'rq2_annotation_quality_by_level.png', dpi=180)
plt.show()

In [ ]:
# RQ2 Plot 3: Case A/B composition (correct vs failure count) by target

fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=5.8, panel_h=4.8)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    case_plot = (
        annotation_quality_viz[annotation_quality_viz['taxonomy_level'] == level]
        .sort_values('total_matches', ascending=False)
        .head(8)
        .sort_values('total_matches', ascending=True)
        .copy()
    )

    if case_plot.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{level}: Case A/B composition')
        ax.set_axis_off()
        continue

    ax.barh(case_plot['target'], case_plot['case_a_present_hateful'], color='#1b9e77', label='Case A (correct)')
    ax.barh(
        case_plot['target'],
        case_plot['case_b_present_nonhateful'],
        left=case_plot['case_a_present_hateful'],
        color='#d95f02',
        label='Case B (failure)',
    )
    ax.set_title(f'{level}: Case A/B composition')
    ax.set_xlabel('Matched count')
    ax.set_ylabel('Target')
    ax.legend(frameon=False, fontsize=8)

fig.suptitle(
    'RQ2 — Case A/B composition by level (self-referential excluded)',
    y=0.99,
    fontsize=14,
    fontweight='semibold',
)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(viz_output_dir / 'rq2_case_ab_composition.png', dpi=180)
plt.show()

In [ ]:
# RQ2 — High-confidence misclassifications (self-referential excluded)

print('=== High-confidence misclassifications (self-referential excluded) ===')

misclassified_forms = form_labeling_viz.copy()
misclassified_forms['total_matches'] = (
    misclassified_forms['case_a_count'] + misclassified_forms['case_b_count']
)
misclassified_forms = misclassified_forms[
    (misclassified_forms['case_b_count'] > misclassified_forms['case_a_count'])
    & (misclassified_forms['total_matches'] >= MIN_MATCHES_FOR_RELIABLE_RATE)
    & (misclassified_forms['labeling_accuracy'] <= 0.20)
].copy()

misclassified_forms = misclassified_forms.sort_values(
    ['taxonomy_level', 'target', 'dogwhistle', 'labeling_accuracy', 'case_b_count'],
    ascending=[True, True, True, True, False],
)

if misclassified_forms.empty:
    print('No high-confidence misclassifications found under current filters.')
else:
    print(f'High-confidence misclassified forms: {len(misclassified_forms)}')
    print(
        misclassified_forms[
            [
                'taxonomy_level',
                'target',
                'dogwhistle',
                'type',
                'case_a_count',
                'case_b_count',
                'total_matches',
                'labeling_accuracy',
            ]
        ].to_string(index=False)
    )

    print('\n=== All matched non-hateful examples for these forms ===')
    all_examples = audit_matches_viz.merge(
        misclassified_forms[['taxonomy_level', 'target', 'dogwhistle']],
        on=['taxonomy_level', 'target', 'dogwhistle'],
        how='inner',
    )
    all_examples = all_examples[all_examples['binary_hate'] == 0].copy()
    all_examples = all_examples.drop_duplicates(
        ['taxonomy_level', 'target', 'dogwhistle', 'text_dedup_key']
    )
    all_examples = all_examples.sort_values(
        ['taxonomy_level', 'target', 'dogwhistle', 'text_dedup_key']
    )
    print(f'Total non-hateful matched examples: {len(all_examples)}')
    for _, row in all_examples.iterrows():
        print(f"- [{row['taxonomy_level']} | {row['target']} | {row['dogwhistle']}] {str(row['text'])}")

---
## RQ3 — Equalized Odds Across Target Groups

> *Do benchmarks satisfy equalized odds across target groups?*

Plots show **disparate-impact (DI) ratios** for coverage and labeling quality between pairs of
target groups within the same taxonomy level.  A DI ratio below 0.8 (the 80% rule) flags
potential disparity.  The summary table (Table 2) flags cells with insufficient data.

In [ ]:
# RQ3 Plot 1: DI-ratio distributions by taxonomy level

fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=5.8, panel_h=4.4)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    level_df = coverage_disparity_viz[coverage_disparity_viz['taxonomy_level'] == level]

    if level_df.empty:
        ax.text(0.5, 0.5, 'No pairwise rows', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{level}: DI distribution')
        ax.set_axis_off()
        continue

    ax.hist(level_df['presence_rate_di_ratio'], bins=12, color='#4c78a8', alpha=0.65, label='presence DI')
    ax.hist(level_df['type_coverage_di_ratio'], bins=12, color='#72b7b2', alpha=0.65, label='type DI')
    ax.axvline(0.8, color='red', linestyle='--', linewidth=1.3, label='80% rule')
    ax.set_title(f'{level}: DI distribution')
    ax.set_xlabel('DI ratio')
    ax.set_ylabel('Pair count')
    if idx == 0:
        ax.legend(frameon=False, fontsize=8)

fig.suptitle(
    'RQ3 — DI-ratio distributions by taxonomy level (self-referential excluded)',
    y=0.99,
    fontsize=14,
    fontweight='semibold',
)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(viz_output_dir / 'rq3_di_ratio_histograms.png', dpi=180)
plt.show()

In [ ]:
# RQ3 Plot 2: Worst DI-ratio pairwise comparisons by taxonomy level

fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=6.4, panel_h=5.0)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    worst_di = coverage_disparity_viz[coverage_disparity_viz['taxonomy_level'] == level].copy()

    if worst_di.empty:
        ax.text(0.5, 0.5, 'No pairwise rows', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{level}: lowest DI comparisons')
        ax.set_axis_off()
        continue

    worst_di['pair'] = worst_di['target_a'] + ' vs ' + worst_di['target_b']
    worst_di['worst_di_ratio'] = worst_di[['presence_rate_di_ratio', 'type_coverage_di_ratio']].min(axis=1)
    worst_di = worst_di.sort_values('worst_di_ratio', ascending=True).head(8)

    colors = ['#cc0000' if v < 0.8 else '#b279a2' for v in worst_di['worst_di_ratio']]
    ax.barh(worst_di['pair'], worst_di['worst_di_ratio'], color=colors)
    ax.axvline(0.8, color='red', linestyle='--', linewidth=1.3)
    ax.set_title(f'{level}: lowest DI comparisons')
    ax.set_xlabel('Worst DI ratio')
    ax.set_xlim(0, 1.05)

fig.suptitle(
    'RQ3 — Lowest DI-ratio pairs by level (red = below 80% rule; self-referential excluded)',
    y=0.99,
    fontsize=14,
    fontweight='semibold',
)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(viz_output_dir / 'rq3_worst_di_comparisons.png', dpi=180)
plt.show()

In [ ]:
# RQ3 Plot 3: Annotation disparity gaps between pairs (labeling-rate gap)

fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=6.4, panel_h=5.0)

for idx, level in enumerate(coarse_levels):
    ax = axes[idx]
    ann_gap = annotation_disparity_viz[annotation_disparity_viz['taxonomy_level'] == level].copy()

    if ann_gap.empty:
        ax.text(0.5, 0.5, 'No pairwise rows', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{level}: labeling-rate gaps')
        ax.set_axis_off()
        continue

    ann_gap['pair'] = ann_gap['target_a'] + ' vs ' + ann_gap['target_b']
    ann_gap = ann_gap.sort_values('labeling_rate_gap_abs', ascending=False).head(8).sort_values('labeling_rate_gap_abs')

    ax.barh(ann_gap['pair'], ann_gap['labeling_rate_gap_abs'], color='#e45756')
    ax.set_title(f'{level}: labeling-rate gaps')
    ax.set_xlabel('Absolute correct-labeling-rate gap')

fig.suptitle(
    'RQ3 — Annotation quality gaps by level (self-referential excluded)',
    y=0.99,
    fontsize=14,
    fontweight='semibold',
)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(viz_output_dir / 'rq3_annotation_labeling_gap.png', dpi=180)
plt.show()

In [ ]:
# RQ3 Plot 4: Cross-level consistency deltas per coarse category

if cross_level_consistency_viz.empty:
    print('cross_level_consistency is empty for this run after self-referential exclusion.')
else:
    fig, axes = make_facet_axes(coarse_levels, ncols=3, panel_w=6.8, panel_h=4.8)

    for idx, level in enumerate(coarse_levels):
        ax = axes[idx]
        level_df = cross_level_consistency_viz[cross_level_consistency_viz['level_from'] == level].copy()

        if level_df.empty:
            ax.text(0.5, 0.5, 'No transitions from this level', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{level}: cross-level deltas')
            ax.set_axis_off()
            continue

        labels = level_df['target'] + ' → ' + level_df['level_to'].astype(str)
        y = np.arange(len(level_df))

        ax.barh(y - 0.2, level_df['presence_rate_delta'], height=0.38, color='#4c78a8', label='presence Δ')
        ax.barh(y + 0.2, level_df['correct_labeling_rate_delta'], height=0.38, color='#54a24b', label='labeling Δ')
        ax.set_yticks(y)
        ax.set_yticklabels(labels)
        ax.axvline(0, color='black', linewidth=1)
        ax.set_title(f'{level}: cross-level deltas')
        ax.set_xlabel('Δ (to − from)')
        if idx == 0:
            ax.legend(frameon=False, fontsize=8)

    fig.suptitle(
        'RQ3 — Cross-level deltas (self-referential excluded)',
        y=0.99,
        fontsize=14,
        fontweight='semibold',
    )
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(viz_output_dir / 'rq3_cross_level_deltas.png', dpi=180)
    plt.show()

In [ ]:
# RQ3 Table 2: Equalized-odds summary — mean DI ratios by (coarse_category_pair × taxonomy_level)
# Cells with fewer than MIN_MATCHES_FOR_RELIABLE_RATE pairs are flagged.

rq3_rows = []
for _, row in coverage_disparity_viz.iterrows():
    ca = row['coarse_a']
    cb = row['coarse_b']
    pair = ' vs '.join(sorted([ca, cb]))
    rq3_rows.append({
        'coarse_pair': pair,
        'taxonomy_level': row['taxonomy_level'],
        'presence_rate_di_ratio': row['presence_rate_di_ratio'],
        'type_coverage_di_ratio': row['type_coverage_di_ratio'],
    })

if rq3_rows:
    rq3_df = pd.DataFrame(rq3_rows)
    rq3_agg = (
        rq3_df.groupby(['coarse_pair', 'taxonomy_level'], sort=True)
        .agg(
            mean_presence_di=('presence_rate_di_ratio', 'mean'),
            mean_type_di=('type_coverage_di_ratio', 'mean'),
            n_pairs=('presence_rate_di_ratio', 'count'),
        )
        .reset_index()
    )
    rq3_agg['mean_presence_di'] = rq3_agg['mean_presence_di'].round(3)
    rq3_agg['mean_type_di'] = rq3_agg['mean_type_di'].round(3)
    rq3_agg['data_flag'] = rq3_agg['n_pairs'].apply(
        lambda n: '⚠ insufficient data' if n < MIN_MATCHES_FOR_RELIABLE_RATE else ''
    )
    rq3_agg['below_80pct_rule'] = (
        (rq3_agg['mean_presence_di'] < 0.8) | (rq3_agg['mean_type_di'] < 0.8)
    )

    print('RQ3 Table 2 — Mean DI ratios by coarse-category pair × taxonomy level')
    print(f'(⚠ = fewer than {MIN_MATCHES_FOR_RELIABLE_RATE} pairs in cell; self-referential excluded)\n')

    # Pivot for compact view
    rq3_pivot = rq3_agg.pivot_table(
        index='coarse_pair',
        columns='taxonomy_level',
        values='mean_presence_di',
        aggfunc='first',
    ).round(3)
    print('Presence-rate DI ratios:')
    print(rq3_pivot.to_string())

    flag_df = rq3_agg[rq3_agg['data_flag'] != ''][['coarse_pair', 'taxonomy_level', 'n_pairs', 'data_flag']]
    if not flag_df.empty:
        print('\nInsufficient-data cells:')
        print(flag_df.to_string(index=False))

    # Export
    rq3_agg.to_csv(viz_output_dir / 'rq3_table2_equalized_odds.tsv', sep='\t', index=False)
    print(f'\nExported → {viz_output_dir / "rq3_table2_equalized_odds.tsv"}')
else:
    print('No coverage disparity rows available for RQ3 table.')

In [ ]:
print(f'Visualization images written to: {viz_output_dir}')
print('\nFiles produced:')
for p in sorted(viz_output_dir.iterdir()):
    print(f'  {p.name}')